In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve
from src.curves.shocks.curve_shocks import CurveShockEngine

from src.instruments.instrument_builder import InstrumentBuilder

In [2]:
# downloading market curves
loader = MarketLoader()

market_curves = loader.market_loader_pipeline()
display(market_curves)

# downloading swap curves
swap_curves = loader.swap_loader_pipeline()
display(swap_curves)

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..


treasury                                            sofr futures  \
                 1M    3M    6M    1Y    2Y    5Y   10Y   30Y    ON      3M   
Date                                                                          
2019-10-01     1.79  1.82  1.81  1.73  1.56  1.51  1.65  2.11  1.88    1.78   
2019-10-02     1.75  1.79  1.75  1.67  1.48  1.43  1.60  2.09  1.85    1.75   
2019-10-03     1.78  1.70  1.66  1.58  1.39  1.34  1.54  2.04  1.84    1.67   
2019-10-04     1.73  1.71  1.65  1.58  1.40  1.34  1.52  2.01  1.82    1.68   
2019-10-07     1.76  1.75  1.73  1.64  1.46  1.38  1.56  2.05  1.83    1.71   
...             ...   ...   ...   ...   ...   ...   ...   ...   ...     ...   
2026-05-25     3.72  3.68  3.79  3.86  4.13  4.27  4.56  5.07  3.55    3.59   
2026-05-26     3.72  3.68  3.80  3.82  4.01  4.19  4.50  5.03  3.63    3.60   
2026-05-27     3.72  3.68  3.79  3.80  4.00  4.17  4.48  5.01  3.63    3.59   
2026-05-28     3.72  3.69  3.79  3.80  3.99  4.15  4.45  4.98  3.62    3.60   
2026-05-29     3.72  3.69  3.79  3.80  3.99  4.15  4.45  4.98  3.62    3.60   

                   estr  
              6M   ESTR  
Date                     
2019-10-01  1.76 -0.549  
2019-10-02  1.71 -0.551  
2019-10-03  1.62 -0.555  
2019-10-04  1.61 -0.553  
2019-10-07  1.69 -0.554  
...          ...    ...  
2026-05-25  3.64  1.931  
2026-05-26  3.65  1.932  
2026-05-27  3.64  1.932  
2026-05-28  3.64  1.933  
2026-05-29  3.64  1.930  

[1739 rows x 12 columns]

usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


usd_ois                                                        \
                1Y    2Y    3Y    4Y    5Y    7Y   10Y   15Y   20Y   30Y   
Date                                                                       
2026-05-29    3.82  3.86  3.85  3.86  3.87  3.93  4.03  4.21  4.28  4.26   

           eur_ois                                                  
                1Y    2Y    3Y    4Y    5Y    7Y   10Y   15Y   30Y  
Date                                                                
2026-05-29    2.41  2.44  2.45  2.48  2.52  2.61  2.77  2.98  3.04

In [3]:
### curve snapshot of a given date
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

display(sofr_snapshot.summary())

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

display(futures_snapshot.summary())

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

display(ois_snapshot.summary())

,Tenor,Years,Rate
0,ON,0.002778,3.62


,Tenor,Years,Rate
0,3M,0.25,3.60
1,6M,0.50,3.64


,Tenor,Years,Rate
0,1Y,1.0,3.82
1,2Y,2.0,3.86
2,3Y,3.0,3.85
3,4Y,4.0,3.86
4,5Y,5.0,3.87
5,7Y,7.0,3.93
6,10Y,10.0,4.03
7,15Y,15.0,4.21
8,20Y,20.0,4.28
9,30Y,30.0,4.26


In [4]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)
display(deposit_instruments)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)
display(future_instruments)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)
display(ois_instruments)

# bootstrap discount curve
engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = deposit_instruments
)

discount_curve.summary()

[DepositInstrument(type=deposit, tenor=ON, rate=3.62)]

[FutureInstrument(type=future, tenor=3M, rate=3.6),
 FutureInstrument(type=future, tenor=6M, rate=3.64)]

[OISInstrument(type=ois_swap, tenor=1Y, rate=3.82),
 OISInstrument(type=ois_swap, tenor=2Y, rate=3.86),
 OISInstrument(type=ois_swap, tenor=3Y, rate=3.85),
 OISInstrument(type=ois_swap, tenor=4Y, rate=3.86),
 OISInstrument(type=ois_swap, tenor=5Y, rate=3.87),
 OISInstrument(type=ois_swap, tenor=7Y, rate=3.93),
 OISInstrument(type=ois_swap, tenor=10Y, rate=4.03),
 OISInstrument(type=ois_swap, tenor=15Y, rate=4.21),
 OISInstrument(type=ois_swap, tenor=20Y, rate=4.28),
 OISInstrument(type=ois_swap, tenor=30Y, rate=4.26)]

,Maturity,DiscountFactor
0,0.002778,0.999899


In [5]:
# testing new bootstrapping engine for generating the discount curve by expanding from earlier maturities
all_instruments = deposit_instruments + future_instruments + ois_instruments

curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

curve.summary()

,Maturity,DiscountFactor
0,0.002778,0.999899
1,0.250000,0.991079
2,0.500000,0.982141
3,1.000000,0.963206
4,2.000000,0.928333
5,3.000000,0.896459
6,4.000000,0.866251
7,5.000000,0.837872
8,7.000000,0.784252
9,10.000000,0.712758


In [6]:
# testing projection curve
projection_curve = ProjectionCurve(discount_curve = curve)

projection_curve.summary()

,Start,End,ForwardRate
0,0.002778,0.25,3.600000
1,0.250000,0.50,3.640000
2,0.500000,1.00,3.931833
3,1.000000,2.00,3.756502
4,2.000000,3.00,3.555514
5,3.000000,4.00,3.487225
6,4.000000,5.00,3.387041
7,5.000000,7.00,3.418517
8,7.000000,10.00,3.343529
9,10.000000,15.00,3.257306


In [7]:
# testing the zero curve
zero_curve = ZeroCurve(discount_curve = curve)

zero_curve.summary()

,Maturity,ZeroRate
0,0.002778,3.619818
1,0.250000,3.584472
2,0.500000,3.604005
3,1.000000,3.748844
4,2.000000,3.718254
5,3.000000,3.643424
6,4.000000,3.589518
7,5.000000,3.537803
8,7.000000,3.471780
9,10.000000,3.386128


In [8]:
### curve shocks
# parallel shift
shocked_curve_parallel = CurveShockEngine.parallel_shift(
    curve = zero_curve,
    parallel_shock_in_bps = 10
)

parallel_up_curve = CurveShockEngine.shock_report(
    curve = zero_curve,
    shocked_curve = shocked_curve_parallel
)

parallel_up_curve 

,Maturity,BaseRate,ShockedRate,ShockBps
0,0.002778,3.619818,3.719818,10.0
1,0.250000,3.584472,3.684472,10.0
2,0.500000,3.604005,3.704005,10.0
3,1.000000,3.748844,3.848844,10.0
4,2.000000,3.718254,3.818254,10.0
5,3.000000,3.643424,3.743424,10.0
6,4.000000,3.589518,3.689518,10.0
7,5.000000,3.537803,3.637803,10.0
8,7.000000,3.471780,3.571780,10.0
9,10.000000,3.386128,3.486128,10.0


In [9]:
# key_rate shift
shocked_curve_key_rate = CurveShockEngine.key_rate_shift(
    curve = zero_curve,
    maturity = 4.0,
    shock_in_bps = 30
)

key_rate_curve = CurveShockEngine.shock_report(
    curve = zero_curve,
    shocked_curve = shocked_curve_key_rate
)

key_rate_curve

,Maturity,BaseRate,ShockedRate,ShockBps
0,0.002778,3.619818,3.619818,0.0
1,0.250000,3.584472,3.584472,0.0
2,0.500000,3.604005,3.604005,0.0
3,1.000000,3.748844,3.748844,0.0
4,2.000000,3.718254,3.718254,0.0
5,3.000000,3.643424,3.643424,0.0
6,4.000000,3.589518,3.889518,30.0
7,5.000000,3.537803,3.537803,0.0
8,7.000000,3.471780,3.471780,0.0
9,10.000000,3.386128,3.386128,0.0


In [10]:
# curve steepener
shocked_curve_steepener = CurveShockEngine.steepener(
    curve = zero_curve
)

steepener_curve = CurveShockEngine.shock_report(
    curve = zero_curve,
    shocked_curve = shocked_curve_steepener
)

steepener_curve

,Maturity,BaseRate,ShockedRate,ShockBps
0,0.002778,3.619818,3.419818,-20.000000
1,0.250000,3.584472,3.387768,-19.670340
2,0.500000,3.604005,3.410635,-19.336976
3,1.000000,3.748844,3.562142,-18.670247
4,2.000000,3.718254,3.544886,-17.336790
5,3.000000,3.643424,3.483391,-16.003334
6,4.000000,3.589518,3.442819,-14.669877
7,5.000000,3.537803,3.404439,-13.336420
8,7.000000,3.471780,3.365085,-10.669506
9,10.000000,3.386128,3.319437,-6.669136


In [11]:
# curve flattener
shocked_curve_flattener = CurveShockEngine.flattener(
    curve = zero_curve
)

flattener_curve = CurveShockEngine.shock_report(
    curve = zero_curve,
    shocked_curve = shocked_curve_flattener
)

flattener_curve

,Maturity,BaseRate,ShockedRate,ShockBps
0,0.002778,3.619818,3.819818,20.000000
1,0.250000,3.584472,3.781175,19.670340
2,0.500000,3.604005,3.797374,19.336976
3,1.000000,3.748844,3.935547,18.670247
4,2.000000,3.718254,3.891622,17.336790
5,3.000000,3.643424,3.803458,16.003334
6,4.000000,3.589518,3.736217,14.669877
7,5.000000,3.537803,3.671168,13.336420
8,7.000000,3.471780,3.578475,10.669506
9,10.000000,3.386128,3.452819,6.669136
